# CAMUS — DRL Contour Refinement: LV_epi (Class 2)

| Variable | Options |
|---|---|
| `CAMUS_CLASS` | `'c1'` (LV_endo) · `'c2'` (LV_epi) · `'c3'` (LA) |
| `AGENT_NAME` | `'DuelingDDQN'` (discrete contour) · `'TD3'` (continuous contour) | `'DuelingDDQN'` (discrete contour) · `'TD3'` (continuous contour) |

Local contour deformation on the lite-U-Net mask; baseline-centred PBRS reward (`dice_hd_pbrs`). Attention U-Net (03) = competitor; lite U-Net (01) = warm-start.

> Run §3 (dry-run) first, then §4 (full).

> ### Phase guide -- which notebook/data for which phase
> This notebook -- which refines the lite U-Net's mask with a DRL agent (CAMUS) -- is the **Phase A** (full-dataset) version. Run it as-is, no changes needed.
>
> | Phase | Data | Notebook |
> |---|---|---|
> | **A** (full data) | full dataset | **this notebook**, unmodified |
> | **B** (low-data regime) | ~150 training images | `notebooks/phaseB/camus/drl/03b_camus_drl_lv_epi.ipynb` |
> | **C** (MSA backbone, smaller subset) | ~75 training images | not yet available -- needs the archived MSA backbone wired into `AGENT_REGISTRY` first (see `docs/EXPERIMENTS.md` section 4) |
>
> Full methodology, exact sizes, and why: **`docs/EXPERIMENTS.md`**.


## 0 · Install + locate package

In [ ]:
import subprocess, sys
from pathlib import Path

init_files = list(Path('/kaggle/input').rglob('iteris/__init__.py'))
if not init_files:
    raise RuntimeError('iteris-pkg not attached.')
PKG_ROOT = init_files[0].parent.parent
subprocess.run(['pip', 'install', '-r', str(PKG_ROOT / 'requirements.txt'),
                '--quiet', '--upgrade'], check=True)
sys.path.insert(0, str(PKG_ROOT))
print(f'✓ Package at: {PKG_ROOT}')

## 1 · Configure — set algorithm here

In [ ]:
# ══════════════════════════════════════════════════════════════════════
CAMUS_CLASS = 'c2'     # 'c1' (LV_endo) | 'c2' (LV_epi) | 'c3' (LA)
AGENT_NAME  = 'DuelingDDQN'   # 'DuelingDDQN' = discrete contour refinement | 'TD3' = continuous contour refinement
# ══════════════════════════════════════════════════════════════════════

_CONFIG_MAP = {'c1': 'camus_drl_c1.yaml',
               'c2': 'camus_drl_c2.yaml',
               'c3': 'camus_drl_c3.yaml'}
assert CAMUS_CLASS in _CONFIG_MAP, f'CAMUS_CLASS must be one of {list(_CONFIG_MAP)}'

from iteris.config import load_drl_class_config, resolve_agent_config, load_config
from iteris.utils  import get_device

cfg_full     = load_drl_class_config(str(PKG_ROOT / 'configs' / 'CAMUS' / 'DRL' / _CONFIG_MAP[CAMUS_CLASS]))
cfg          = resolve_agent_config(cfg_full, AGENT_NAME)

# ============================================================================
# ITERIS DRL CONFIG -- one consolidated in-memory cfg patch (see docs/
# EXPERIMENTS.md / SKILLS.md for full rationale history). Refines the
# ATTENTION U-Net (boundary-error regime a contour agent can fix -- not the
# lite net's structural errors) with a dense per-point boundary reward, a
# GT-free routing gate, fine-step mechanics, local per-sector perception, and
# a shared GT-oracle BC warm-start for both agents. Shared YAML untouched.
# ============================================================================
cfg['baseline_cfg_name'] = 'CAMUS/camus.yaml'   # -> camus_best.pt (Phase A) / _lf<pct>_best.pt (Phase B, via label_frac below)

cfg.update({
    # Reward -- dense per-control-point distance-to-GT-boundary (not global
    # Dice): self-regularising (pushing an already-correct point earns negative
    # reward), so the agent naturally stops instead of drifting/overshooting.
    'reward_mode':          'contour_boundary',
    'reward_clip':           10.0,   # deltas are in pixels, not Dice units
    'reward_step_penalty':   0.0,    # DuelingDDQN's YAML default (0.05) drove an 88%-STOP collapse; 0 for both agents

    # GT-free routing gate -- train/eval only on cases whose init mask is a
    # single dominant, plausibly-sized component (contour-fixable). Excluded
    # cases are ROUTED to the raw U-Net mask at eval, never GT-dropped.
    'refinable_gate':          True,
    'refinable_min_cc_frac':   0.004,
    'refinable_min_dominance': 0.5,

    # U-Net confidence gate -- only valid where prob_map is graded (USABLE),
    # not near-binary (INERT, where it clamps edits to ~0).
    'uncertainty_gate': False,

    # Fine-step mechanics -- step size must be finer than the residual
    # boundary error (~2.5px) or every push overshoots (verified: identical
    # toward-GT pushes converge at disp_px=0.5 but degrade at disp_px=2.0).
    'disp_px':              0.5,
    'max_steps':             25,     # continuous (TD3): moves every sector each step; bumped for discrete below
    'auto_smooth_lambda':    0.1,    # per-step Laplacian smoothing (continuous has no explicit SMOOTH action)
    'curriculum_max_steps':  False,  # every sample gets the full step budget
    'disable_auto_stop':     True,   # the YAML's stop_eps_dice was tuned for disp_px=2.0; falsely 'converges' after ~3 fine steps

    # Perception -- per-sector LOCAL features (DeepSnake/MARL mechanism), not
    # one globally-pooled vector deciding every sector's push.
    'spatial_head': True,

    # BC warm-start -- GT-oracle imitation for BOTH agents (fair continuous-
    # vs-discrete comparison; TD3's actor starts near-identity, DuelingDDQN's
    # Q-net starts random -- neither escapes that without a warm start).
    'bc_warm_start':      True,
    'bc_demo_episodes':   80,
    'bc_demo_max_steps':  12,   # bumped to 30 below for discrete (moves one sector/step)

    # Ablations -- unproven / not applicable here; left off.
    'directional_state': False,
    'topology_filter':   False,
})

# Discrete moves ONE sector/step (continuous moves all of them), so it needs a
# higher max_steps CAP + deeper BC demos to reach/demonstrate a distributed
# correction. It self-terminates via its learned STOP, so this is a ceiling,
# not a fixed length (research: single-element-per-step RL refinement uses
# tens-to-hundreds of steps/episode, e.g. mesh-refinement RL 200-400).
if str(AGENT_NAME).upper() != 'TD3':
    cfg['max_steps'] = 50
    cfg['bc_demo_max_steps'] = 30

print(f"[config] base={cfg['baseline_cfg_name']} agent={AGENT_NAME} "
      f"reward={cfg['reward_mode']} gate={cfg['uncertainty_gate']} "
      f"refinable_gate={cfg['refinable_gate']} disp_px={cfg['disp_px']} "
      f"max_steps={cfg['max_steps']} spatial_head={cfg['spatial_head']} "
      f"bc_warm_start={cfg['bc_warm_start']}")

baseline_cfg = load_config(str(PKG_ROOT / 'configs' / cfg['baseline_cfg_name']))

cfg['checkpoint_dir'] = '/kaggle/working'

# Auto-detect CAMUS data root
camus_roots = [p for p in Path('/kaggle/input').rglob('CAMUS') if p.is_dir()]
if camus_roots:
    baseline_cfg['data_root'] = str(camus_roots[0])

# Auto-detect the lite-vs-attention checkpoint by deriving the expected
# filename from baseline_cfg (model_suffix handles _lite_unet vs '' default).
from iteris.utils import model_suffix
_ckpt_name  = f"{baseline_cfg['dataset'].lower()}{model_suffix(baseline_cfg)}_best.pt"
_ckpt_cands = list(Path('/kaggle/input').rglob(_ckpt_name))
if _ckpt_cands:
    cfg['baseline_checkpoint'] = str(_ckpt_cands[0])
else:
    raise FileNotFoundError(
        f'Baseline checkpoint {_ckpt_name!r} not found under /kaggle/input. '
        f'Attach the dataset containing it, or override cfg["baseline_checkpoint"] manually.')

print(f'Agent           : {cfg["agent_type"]}')
print(f'Target class    : {cfg["target_class"]} ({cfg["class_name"]})')
print(f'Reward mode     : {cfg["reward_mode"]}  '
      f'(α={cfg.get("reward_alpha","-")} β={cfg.get("reward_beta","-")} '
      f'hd_norm={cfg.get("hd_norm","-")})')
print(f'CAMUS data root : {baseline_cfg["data_root"]}')
print(f'Baseline ckpt   : {cfg["baseline_checkpoint"]}')
print(f'Train steps     : {cfg["train_steps"]}')
print(f'Device          : {get_device()}')

## 2 · Warm-start — pre-compute U-Net init masks
~5 min for CAMUS. Caches U-Net predictions so the DRL loop never re-runs the backbone.

In [ ]:
from iteris.warm_start import precompute_init_masks
import numpy as np

train_samples, val_samples, test_samples = precompute_init_masks(
    baseline_cfg        = baseline_cfg,
    baseline_checkpoint = cfg['baseline_checkpoint'],
    target_class        = cfg['target_class'],
    min_area_fraction   = cfg.get('min_area_fraction', 0.01),
)
print(f'\nSamples — train: {len(train_samples)}  val: {len(val_samples)}  test: {len(test_samples)}')

init_dices = [float((s['init_mask'] & s['gt_mask']).sum() * 2 /
                    (s['init_mask'].sum() + s['gt_mask'].sum() + 1e-6))
              for s in val_samples]
print(f'U-Net init Dice on val ({cfg["class_name"]}): '
      f'mean {np.mean(init_dices):.4f}  median {np.median(init_dices):.4f}')

## 3 · OPTIONAL — Dry-run sanity check (~3 min)
Self-contained `deepcopy(cfg)` — does NOT mutate `cfg`, `train_samples`, or `val_samples`.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
RUN_DRY_RUN = False   # Set True for a quick ~2-3 min sanity check before full training
# ══════════════════════════════════════════════════════════════════════

if RUN_DRY_RUN:
    import copy
    from iteris.drl_training import run_drl_training

    # Short sanity run on a small train/val slice — same loop as the full run.
    _dry = copy.deepcopy(cfg)
    _dry.update(dict(train_steps=600, prefill_steps=500, buffer_size=3000,
                     eval_every=200, epsilon_decay_steps=400, batch_size=32))
    _dry_result = run_drl_training(_dry, train_samples[:30], val_samples[:10])
    print(f'\n✓ Dry run passed. Best val score (meaningless at 600 steps): {_dry_result["best_dice"]:.4f}')
    print(f'  cfg["train_steps"] still = {cfg["train_steps"]}  (unchanged)')
    print('  → Safe to run §4 below for the real training run.')
else:
    print('Dry run skipped (RUN_DRY_RUN = False). Proceed directly to §4.')


## 4 · Full training

**DuelingDDQN** (discrete contour) and **TD3** (continuous contour): ~1.5–3 hr on T4 (contour rasterisation is the bottleneck). Run this **or** §3 — not both.

The best-val checkpoint is saved to `/kaggle/working` whenever the score improves, so a Kaggle disconnect only costs time since the last improvement.

In [ ]:
# ── Diagnostic / quota controls ───────────────
# Split a long run into cheaper chunks so a broken config is caught early
# instead of after a full ~6h/50k run:
#   1) First pass:  TRAIN_STEPS = 20000, RESUME = False  (a ~1/3-cost probe).
#      Inspect the curve (§9) + value-floored test eval (§10). If final/
#      best-seen trends up and beats baseline, continue WITHOUT redoing it:
#   2) Continue:    TRAIN_STEPS = 50000, RESUME = True   (resumes from the
#      *_last.pt checkpoint, runs only the remaining 40k; epsilon/optimizer
#      state carry over). The replay buffer re-prefills cold but eval is
#      greedy, so the resumed curve stays continuous.
# Set TRAIN_STEPS = None to use the config's full train_steps in one shot.
TRAIN_STEPS = 30000     # 30k now; RESUME=True + TRAIN_STEPS=60000 to continue from *_last.pt
RESUME      = False     # True = continue from *_last.pt (set TRAIN_STEPS to the NEW total)
if TRAIN_STEPS is not None:
    cfg['train_steps'] = TRAIN_STEPS
cfg['resume'] = RESUME
print(f"[run] train_steps={cfg['train_steps']}  resume={cfg['resume']}")
# ────────────────

from iteris.drl_training import run_drl_training

result    = run_drl_training(cfg, train_samples, val_samples)
agent     = result['agent']
history   = result['history']     # pandas DataFrame
best_dice = result['best_dice']

print(f"\n✓ Training complete — {cfg['agent_type']} on {cfg['class_name']}")
print(f"  Best val final-Dice : {best_dice:.4f}")
print(f"  Checkpoint          : {result['checkpoint']}")


## 5 · Visualisation setup
Defines `replay_episode()` and `ENV_KW`. Used by all cells below.

In [ ]:
import matplotlib.pyplot as plt   # needed by the plot cells below (14/16/18)
from iteris.refinement_viz import (
    refinement_env_kwargs, refinement_env_cls, build_replays, plot_comparison,
    plot_playback, plot_behaviour, evaluate_testset)

ENV_KW  = refinement_env_kwargs(cfg)
ENV_CLS = refinement_env_cls(cfg)   # SegmentationEnv, or ContourRefineEnv for contour/TD3 -- auto-detection
                                     # by agent.num_actions silently picks the WRONG env for continuous
                                     # agents (TD3 has no num_actions), so pass it explicitly.
N_VIZ   = 8
replays = build_replays(agent, val_samples, ENV_KW, n_viz=N_VIZ, seed=0, env_cls=ENV_CLS)

CLASS_NAME = cfg.get('class_name', '')
print(f'Built {len(replays)} greedy replays for visualisation.')

## 6 · Sample comparison — best / median / worst

In [ ]:
import matplotlib.pyplot as plt
fig = plot_comparison(
    replays, baseline_cfg, cfg,
    class_idx=cfg.get('target_class', 1), class_name=CLASS_NAME,
    out_path=f"/kaggle/working/{cfg['dataset'].lower()}_{cfg['agent_type'].lower()}_comparison.png")
plt.show()


## 7 · Iteration playback — all steps on best-gain sample
Green contour = GT boundary. Pay attention to how the apex is handled step-by-step.

In [ ]:
import matplotlib.pyplot as plt
# Playback the BEST-gain episode step-by-step
fig = plot_playback(
    replays[-1], cfg, class_name=CLASS_NAME,
    out_path=f"/kaggle/working/{cfg['dataset'].lower()}_{cfg['agent_type'].lower()}_playback.png")
plt.show()


## 8 · Behaviour analysis — trajectories + action distribution

In [ ]:
import matplotlib.pyplot as plt
fig = plot_behaviour(
    replays, cfg, class_name=CLASS_NAME,
    out_path=f"/kaggle/working/{cfg['dataset'].lower()}_{cfg['agent_type'].lower()}_behaviour.png")
plt.show()


## 9 · Learning curves

In [ ]:
import matplotlib.pyplot as plt

# history is a DataFrame with columns: step, init_dice_mean, final_dice_mean,
# best_dice_mean, final_hd95_mean, delta_dice_mean
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['step'], history['init_dice_mean'],  label='Init Dice (U-Net)', ls='--', alpha=0.6, color='gray')
ax1.plot(history['step'], history['final_dice_mean'], label=f'Final Dice ({cfg["agent_type"]})', color='C0')
if 'best_dice_mean' in history.columns:
    ax1.plot(history['step'], history['best_dice_mean'], label='Best-seen Dice (ceiling)', color='C2', alpha=0.7)
ax1.set(xlabel='Training step', ylabel='Mean Val Dice',
        title=f"{cfg['dataset']} — {cfg['class_name']} refinement curve")
ax1.legend(); ax1.grid(alpha=0.3)

delta = history['final_dice_mean'] - history['init_dice_mean']
ax2.plot(history['step'], delta, color='C0')
ax2.axhline(0, color='k', lw=0.8)
ax2.fill_between(history['step'], delta, 0, where=(delta > 0), alpha=0.15, color='green')
ax2.fill_between(history['step'], delta, 0, where=(delta < 0), alpha=0.15, color='red')
ax2.set(xlabel='Training step', ylabel='Δ Dice (final − init)',
        title=f'Refinement gain — {cfg["agent_type"]} ({cfg["class_name"]})')
ax2.grid(alpha=0.3)

plt.suptitle(f"{cfg['dataset']} {cfg['agent_type']} — {cfg['class_name']} learning curves")
plt.tight_layout()
out = f"/kaggle/working/{cfg['dataset'].lower()}_{cfg['agent_type'].lower()}_curves.png"
plt.savefig(out, dpi=150); plt.show(); print(f'Saved to {out}')


## 10 · Test-set evaluation + summary JSON

In [ ]:
import json
test_metrics = evaluate_testset(agent, test_samples, ENV_KW, env_cls=ENV_CLS)
print(json.dumps(test_metrics, indent=2))

summary = {**test_metrics,
           'agent_type':    cfg['agent_type'],
           'target_class':  cfg['target_class'],
           'class_name':    cfg['class_name'],
           'dataset':       cfg['dataset'],
           'paradigm':      'local_mask_refinement'}
out = f"/kaggle/working/{cfg['dataset'].lower()}_{cfg['agent_type'].lower()}_test_results.json"
with open(out, 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved to', out)
print(f"\nBaseline (U-Net) Dice: {test_metrics['init_dice_mean']:.4f}")
print(f"Refined  (agent)  Dice: {test_metrics['final_dice_mean']:.4f}  "
      f"(Δ {test_metrics['delta_dice_mean']:+.4f})")
print(f"Best-seen ceiling Dice: {test_metrics['best_dice_mean']:.4f}")
